In [ ]:
import pandas as pd
import numpy as np
import os
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import joblib
import warnings

from sklearn.model_selection import train_test_split, GroupKFold, ParameterGrid, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neural_network import MLPClassifier

import tensorflow as tf
from tensorflow import keras

import utils

warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
sampling_rate = 100
n_splits = 3
pipe_name = 'imu_extractor'

acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

linear_edges = np.arange(0, 51, 10)
log_edges = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

some_subjects = ['SUBJ_059520', 'SUBJ_020948', 'SUBJ_040282', 'SUBJ_052342', 'SUBJ_032165']

model_run_name = 'neural_network.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

In [ ]:
train_df = raw_train_df.set_index('row_id')
multiple_gesture_df = train_df[train_df['sequence_id'].isin(['SEQ_000007', 'SEQ_000008', 'SEQ_000013'])]
train_sample_df, test_sample_df = utils.sample_balanced_split(train_df, train_pct=0.6, test_pct=0.2)

In [ ]:
num_pattern = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation']
normal_cols = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        ('feature_num_cols', StandardScaler(), make_column_selector(pattern=num_pattern)),
        ('subject_num_cols', StandardScaler(), utils.existing_cols(suspect_cols)),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), utils.existing_cols(cat_cols)),
        ('normal_cols', 'passthrough', utils.existing_cols(normal_cols)),
        ('ordinal_cols', 'passthrough', utils.existing_cols(ordinal_cols))
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

custom_extractor = utils.ImuExtractor(
    subject_df=train_demo_df,
    imu_sensor_list=['acc_x', 'acc_y', 'acc_z'],
    rotation_sensor_list=['rot_x', 'rot_y', 'rot_z']
)

nn_clf = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    learning_rate_init=0.001,
    max_iter=200,
    validation_fraction=0.1,
    random_state=42,
    verbose=False
)

In [ ]:
importlib.reload(utils)

sliced_df = train_sample_df.copy(deep=True)

param_grid = {
    f'{pipe_name}__imu_sensor_list': [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain': ['acceleration', 'velocity', 'displacement'],
    f'{pipe_name}__combine_imu_axes': [True],
    f'{pipe_name}__sampling_rate': [100],

    f'{pipe_name}__rotation_sensor_list': [None],
    f'{pipe_name}__combine_rot_axes': [True],

    f'{pipe_name}__thermopile_sensor_list': [None],
    f'{pipe_name}__tof_sensor_list': [None],

    f'{pipe_name}__dc_offset': [2.0],
    f'{pipe_name}__band_edges': [None],
    f'{pipe_name}__category_data': [True],
    f'{pipe_name}__segmentation': ['phase', 'window'],

    'classifier__estimator__hidden_layer_sizes': [(32,)],
    'classifier__estimator__activation': ['relu'],
    'classifier__estimator__learning_rate_init': [0.001],
    'classifier__estimator__max_iter': [200],
    'classifier__estimator__alpha': [1], 
}

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(nn_clf, custom_extractor, mode=None))
])


y = sliced_df[['sequence_id', 'gesture']]
groups = sliced_df['sequence_id']

n_fits = len(list(ParameterGrid(param_grid))) * n_splits
pbar = tqdm(total=n_fits, desc="Grid Search Fits")

def tqdm_scorer(estimator, X, y):
    pbar.update(1)
    return estimator.score(X, y)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    scoring=tqdm_scorer,
    verbose=0,
    n_jobs=1,
    return_train_score=True
)

grid_search.fit(sliced_df, y, groups=groups)
pbar.close()

In [ ]:
grid_search.cv_results_

In [ ]:
pd.DataFrame(grid_search.cv_results_).T

In [ ]:
# Permutation Importance
from sklearn.inspection import permutation_importance

# Get best model
best_model = grid_search.best_estimator_

# Transform sliced_df (same data used in grid search)
X_transformed = best_model.named_steps['imu_extractor'].transform(sliced_df)
X_preprocessed = best_model.named_steps['preprocessor'].transform(X_transformed)

# Get labels
y_seq = sliced_df.groupby('sequence_id')['gesture'].first()
y_aligned = y_seq.reindex(X_preprocessed.index)

# Permutation importance
perm_importance = permutation_importance(
    best_model.named_steps['classifier'].estimator_,
    X_preprocessed, y_aligned,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': perm_importance.importances_mean
}).sort_values('importance', ascending=False)

# Plot top 20
plt.figure(figsize=(10, 8))
plt.barh(feat_imp_df['feature'].head(20), feat_imp_df['importance'].head(20))
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (Permutation)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Save
feat_imp_df.to_csv(model_run_folder_name + 'neural_network_feature_importance.csv', index=False)
print(feat_imp_df.head(20))

In [ ]:
model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(model_run_folder_name + 'neural_network_results.csv', index=False)

print(f"Best score: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")